In [1]:
%load_ext autoreload
%autoreload 2

from backend.app.services.data_services import (
    get_available_indicators,
    get_available_strategies,
    get_strategies_metadata,
    get_available_tickers,
    _read_csv,
    fetch_data_to_df,
    get_indicators_metadata,
    _load_and_resample_data
)
import importlib
import backend.app.core.Strategies as st

import backend.app.services.cache_service as cs
import backend.app.services.data_services as ds
import backend.app.services.plotting_services as ps
import backend.app.services.backtest_services as bs

# importlib.reload(cs)
# importlib.reload(ds)
# importlib.reload(ps)
# importlib.reload(bs)

import asyncio
import pandas as pd
import numpy as np
import vectorbt as vbt

In [2]:
# ticker_name = "TATA CONSULTANCY SERVICES"
ticker_name = "SOLUSD"
df = await ds._read_csv(ticker_name=ticker_name)
print(df.head(2))
print(df.tail(2))
await cs.set_data(df=df, ticker= ticker_name)

                       open     high     low   close  volume
date_time                                                   
2025-01-01 00:00:00  195.32  195.370  195.26  195.34      91
2025-01-01 00:01:00  195.45  195.762  195.44  195.66     452
                        open     high      low    close  volume
date_time                                                      
2025-12-20 23:59:00  125.900  125.900  125.882  125.893     746
2025-12-21 00:00:00  125.882  125.923  125.882  125.923      67


{'message': 'Data loaded successfully with key: data:SOLUSD'}

In [3]:
resolution = "15m"
start_date = "01/01/2025 09:15:00"
end_date = "30/10/2025 09:15:00"

df = await ds._load_and_resample_data(ticker_name, resolution, start_date, end_date)
print(df.shape)
price = df['close']
df.head()

(28993, 5)


d:\OneDrive - iitgn.ac.in\Desktop\HedgeOne-Quant\backend\app\services\data_services.py:54: FutureWarning:

'T' is deprecated and will be removed in a future version, please use 'min' instead.



,open,high,low,close,volume
2025-01-01 09:15:00,190.390,190.430,189.949,190.370,961
2025-01-01 09:30:00,190.249,190.280,189.568,189.990,1032
2025-01-01 09:45:00,189.980,190.110,189.629,189.740,800
2025-01-01 10:00:00,189.779,189.779,189.257,189.610,1404
2025-01-01 10:15:00,189.630,189.780,189.300,189.408,1600


In [29]:
import pandas as pd
import plotly.graph_objects as go

# Assuming df is loaded
df.index = pd.to_datetime(df.index)

# Calculate SMA 9 and SMA 12
df['SMA9'] = df['close'].rolling(9).mean()
df['SMA12'] = df['close'].rolling(26).mean()

# Filter out rows where 'close' is NaN
df_filtered = df.dropna(subset=['close'])

# Create candlestick chart
fig = go.Figure()

# Candles
fig.add_trace(go.Candlestick(
    x=df_filtered.index,
    open=df_filtered['open'],
    high=df_filtered['high'],
    low=df_filtered['low'],
    close=df_filtered['close'],
    increasing_line_color='green',
    decreasing_line_color='red',
    name='Candlestick'
))

# SMA lines
fig.add_trace(go.Scatter(
    x=df_filtered.index, 
    y=df_filtered['SMA9'], 
    mode='lines', 
    line=dict(color='blue', width=1.5), 
    name='SMA9'
))
fig.add_trace(go.Scatter(
    x=df_filtered.index, 
    y=df_filtered['SMA12'], 
    mode='lines', 
    line=dict(color='orange', width=1.5), 
    name='SMA12'
))

# Layout
fig.update_layout(
    template='plotly_dark',
    xaxis_title='Date',
    yaxis_title='Price',
    xaxis_rangeslider_visible=False,
    hovermode='x unified',
    title='TradingView-style Candlestick Chart'
)

# Save as HTML
fig.write_html("candlestick_chart.html", auto_open=True)


In [4]:
import pandas as pd
import plotly.graph_objects as go

# Assuming df is loaded
df.index = pd.to_datetime(df.index)

# Calculate SMA 9 and SMA 12
df['SMA9'] = df['close'].rolling(9).mean()
df['SMA12'] = df['close'].rolling(26).mean()

# Filter out rows where 'close' is NaN
df_filtered = df.dropna(subset=['close'])

# Create candlestick chart
fig = go.Figure()

# Candles
fig.add_trace(go.Candlestick(
    x=df_filtered.index,
    open=df_filtered['open'],
    high=df_filtered['high'],
    low=df_filtered['low'],
    close=df_filtered['close'],
    increasing_line_color='green',
    decreasing_line_color='red',
    name='Candlestick'
))

# SMA lines
fig.add_trace(go.Scatter(
    x=df_filtered.index,
    y=df_filtered['SMA9'],
    mode='lines',
    line=dict(color='blue', width=1.5),
    name='SMA9'
))
fig.add_trace(go.Scatter(
    x=df_filtered.index,
    y=df_filtered['SMA12'],
    mode='lines',
    line=dict(color='orange', width=1.5),
    name='SMA12'
))

# Layout for interactivity
fig.update_layout(
    template='plotly_dark',
    title='TradingView-style Candlestick Chart',
    xaxis_title='Date',
    yaxis_title='Price',
    hovermode='x unified',
    xaxis=dict(
        rangeslider=dict(visible=True),  # Enable range slider
        type='date',
        showline=True,
        showgrid=True,
        showticklabels=True,
        rangeselector=dict(
            buttons=list([
                dict(count=1, label="1m", step="month", stepmode="backward"),
                dict(count=3, label="3m", step="month", stepmode="backward"),
                dict(count=6, label="6m", step="month", stepmode="backward"),
                dict(count=1, label="YTD", step="year", stepmode="todate"),
                dict(count=1, label="1y", step="year", stepmode="backward"),
                dict(step="all")
            ])
        )
    ),
    yaxis=dict(fixedrange=False),  # Allow zoom on Y-axis
)

# Save and open HTML
fig.write_html("candlestick_chart_interactive.html", auto_open=True)


In [1]:
print("Hello")

Hello


In [5]:
import math as m

count = 0
n = 10
for i in range(1,n+1):
    count += (i**2)

print(count)
print(m.sqrt(count)/n)

385
1.9621416870348583
